# 📏 Day 05a: KNN & Naive Bayes — Simple but Powerful

---

## 🎯 What You'll Master Today
- KNN: distance metrics, K selection, weighted KNN
- Naive Bayes: Bayes theorem, Gaussian/Multinomial NB
- Both from scratch + sklearn

**Goal:** Implement KNN classifier from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  KNN — K-NEAREST NEIGHBORS                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Algorithm:                                                       ║
║    1. Compute distance to all training points                   ║
║    2. Find K closest neighbors                                  ║
║    3. Majority vote (classification)                             ║
║       or average (regression)                                    ║
║                                                                   ║
║  Lazy learner: no training phase, all work at predict time      ║
║                                                                   ║
║  NAIVE BAYES — P(class|features)                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Bayes: P(y|X) = P(X|y) · P(y) / P(X)                         ║
║  Naive: Features are conditionally independent                  ║
║  P(X|y) = P(x₁|y) · P(x₂|y) · ... · P(xₙ|y)                ║
║                                                                   ║
║  Gaussian: P(xᵢ|y) is normal distribution                      ║
║  Multinomial: P(xᵢ|y) from counts (text classification)       ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np
from collections import Counter

# ============================================================
# Distance Metrics
# ============================================================

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Euclidean: √(Σ(aᵢ - bᵢ)²)
euclidean = np.sqrt(np.sum((a - b) ** 2))
print(f"Euclidean: {euclidean:.4f}")
print(f"NumPy:     {np.linalg.norm(a - b):.4f}")

# Manhattan: Σ|aᵢ - bᵢ|
manhattan = np.sum(np.abs(a - b))
print(f"Manhattan: {manhattan}")

# Minkowski (general): (Σ|aᵢ - bᵢ|^p)^(1/p)
# p=1 → Manhattan, p=2 → Euclidean

print("\n💡 Euclidean = default. Manhattan better for high dimensions.")

In [ ]:
# ============================================================
# ⭐ KNN Classifier — From Scratch
# ============================================================

class KNNScratch:
    def __init__(self, k=3):
        self.k = k
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def _predict_one(self, x):
        # Compute distances to all training points
        distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
        
        # Get K nearest indices
        k_indices = np.argsort(distances)[:self.k]
        k_labels = self.y_train[k_indices]
        
        # Majority vote
        return Counter(k_labels).most_common(1)[0][0]
    
    def predict(self, X):
        return np.array([self._predict_one(x) for x in X])

# Test
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=200, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Always scale for KNN!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNNScratch(k=5)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)
print(f"KNN (scratch) accuracy: {np.mean(y_pred == y_test):.3f}")

In [ ]:
# ============================================================
# K Selection — Finding the Right K
# ============================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

k_values = range(1, 21)
scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_scores = cross_val_score(knn, X_train_s, y_train, cv=5)
    scores.append(cv_scores.mean())

best_k = k_values[np.argmax(scores)]
print(f"Best K={best_k} with CV accuracy={max(scores):.3f}")
print("\nK values vs accuracy:")
for k, s in zip(k_values, scores):
    bar = '█' * int(s * 40)
    marker = ' ← best' if k == best_k else ''
    print(f"  K={k:2d}: {s:.3f} {bar}{marker}")

print("\n💡 Small K → overfit (noisy). Large K → underfit (smooth).")
print("   Rule of thumb: K = √n, use odd K for binary classification.")

In [ ]:
# ============================================================
# Naive Bayes — The Math Then Code
# ============================================================

class GaussianNBScratch:
    """Gaussian Naive Bayes from scratch."""
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.params = {}  # {class: (mean, var, prior)}
        
        for c in self.classes:
            X_c = X[y == c]
            self.params[c] = {
                'mean': X_c.mean(axis=0),
                'var': X_c.var(axis=0) + 1e-9,  # add epsilon for stability
                'prior': len(X_c) / len(X)
            }
        return self
    
    def _gaussian_pdf(self, x, mean, var):
        """P(x|class) using Gaussian distribution."""
        return np.exp(-0.5 * ((x - mean) ** 2) / var) / np.sqrt(2 * np.pi * var)
    
    def predict(self, X):
        predictions = []
        for x in X:
            posteriors = []
            for c in self.classes:
                p = self.params[c]
                # Log posterior = log prior + Σ log P(xᵢ|c)
                log_prior = np.log(p['prior'])
                log_likelihood = np.sum(np.log(self._gaussian_pdf(x, p['mean'], p['var'])))
                posteriors.append(log_prior + log_likelihood)
            predictions.append(self.classes[np.argmax(posteriors)])
        return np.array(predictions)

# Test
nb = GaussianNBScratch()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print(f"Gaussian NB (scratch) accuracy: {np.mean(y_pred_nb == y_test):.3f}")

In [ ]:
# ============================================================
# sklearn Naive Bayes Variants
# ============================================================

from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

# Gaussian NB (continuous features)
gnb = GaussianNB()
gnb.fit(X_train, y_train)
print(f"GaussianNB:    {gnb.score(X_test, y_test):.3f}")

# Compare with KNN
knn_sk = KNeighborsClassifier(n_neighbors=5)
knn_sk.fit(X_train_s, y_train)
print(f"KNN (k=5):     {knn_sk.score(X_test_s, y_test):.3f}")

print("\nNB Variants:")
print("  GaussianNB:    continuous features (numeric data)")
print("  MultinomialNB: count data (text classification, bag of words)")
print("  BernoulliNB:   binary features (word presence/absence)")

print("\n💡 NB is surprisingly good for text classification!")
print("   Despite the 'naive' independence assumption, it works well in practice.")

---

## 🗺️ KNN & Naive Bayes Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  KNN vs NAIVE BAYES — When to Use What                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  KNN:                                                            ║
║    ✓ Small dataset, non-linear boundaries                       ║
║    ✓ No training time (lazy learner)                            ║
║    ✗ Slow prediction (compute all distances)                    ║
║    ✗ Curse of dimensionality (distances meaningless in high-D)  ║
║    MUST: Scale features, choose K via CV                       ║
║                                                                   ║
║  NAIVE BAYES:                                                    ║
║    ✓ Text classification, spam filtering                        ║
║    ✓ Very fast (training = counting)                            ║
║    ✓ Works with small data                                      ║
║    ✗ Independence assumption often violated                     ║
║    NOTE: Still works well despite 'naive' assumption            ║
║                                                                   ║
║  GAUSSIAN: numeric features                                     ║
║  MULTINOMIAL: word counts (TF-IDF, BoW)                        ║
║  BERNOULLI: binary features                                      ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | KNN time complexity? | Training: O(1). Prediction: O(nd) per query (n=samples, d=features). Use KD-tree for speedup |
| 2 | How to choose K? | Cross-validation. Rule of thumb: √n. Use odd K for binary. Small K=overfit, large K=underfit |
| 3 | Why 'naive' in Naive Bayes? | Assumes feature independence given the class. Rarely true but works anyway |
| 4 | When does NB outperform other models? | Text classification, small datasets, when independence assumption roughly holds |
| 5 | KNN curse of dimensionality? | In high-D, all points are nearly equidistant. Distances become meaningless. Use PCA first |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • KNN = lazy learner, O(nd) at predict time            │
## │  • Always scale features for KNN                        │
## │  • Choose K via cross-validation                        │
## │  • NB = fast, good for text, works despite assumption   │
## │  • Use log probabilities to avoid underflow             │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chapter 04 — sklearn Pipelines: Preprocessing + Model Selection**